# NOTEBOOK DEEPSEEK TO MEASURE LOG DRIFT

This notebook evaluates the DeepSeek LLM previously trained to changes of different characteristics to see how the LLM works on log drift.

## 1. IMPORTS AND SETUP
Import the required python libraries.

In [1]:
# Core modules for file handling and JSON
import os
import json
# PyTorch for model inference
import torch
# Data manipulation and progress tracking
import pandas as pd
from tqdm import tqdm
# Hugging Face datasets for structured input
from datasets import Dataset
# Evaluation metrics for classification tasks
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
# For default dictionary behavior
from collections import defaultdict
# Visualization libraries
import seaborn as sns
import matplotlib.pyplot as plt
# Unsloth: optimized LLM loading
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

/home/jorge/TFM/threatlogllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
# Configuration for the model and data
model_path = "../../fine-tuned-model-attacks/deepseek_finetuned_model_multi"
file_path = "../../../../../data/prompts/multiclass_instructions_for_inference_more_samples.jsonl"
max_seq_length = 2048
use_4bit = True
save_csv = True
csv_output_path = "../../outputs/results_deepseek_multiclass_greater.csv"

In [3]:
# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
model, _ = FastLanguageModel.from_pretrained(
    model_name=model_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=use_4bit,
)
FastLanguageModel.for_inference(model)
model.eval()

==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA GeForce RTX 4060 Laptop GPU. Max memory: 7.996 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.2.15 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

## MEASURE LOG DRIFT - 1K SAMPLES

In [26]:
import json
import pandas as pd
attack_types = [
    "Normal", "DDoS_UDP", "DDoS_ICMP", "SQL_injection", "Password",
    "Vulnerability_scanner", "DDoS_TCP", "DDoS_HTTP", "Uploading", "Backdoor",
    "Port_Scanning", "XSS", "Ransomware", "MITM", "Fingerprinting"
]

attack_type_to_id = {a: i for i, a in enumerate(attack_types)}
id_to_attack_type = {i: a for i, a in enumerate(attack_types)}

def label_to_multiclass(text):
    text = text.lower().strip()
    for attack in attack_types:
        if attack.lower() in text:
            return attack_type_to_id[attack]
    return attack_type_to_id["Normal"]

rows = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line.strip())
        full_prompt = obj["Prompt"]

        if "### Response:" in full_prompt:
            input_text, response_part = full_prompt.split("### Response:", maxsplit=1)
            input_text = input_text.strip()
            label_id = label_to_multiclass(response_part.strip())
        else:
            input_text = full_prompt.strip()
            label_id = attack_type_to_id["Normal"]

        rows.append({
            "input_text": input_text,
            "true_label_id": label_id,
            "true_label": id_to_attack_type[label_id],
        })

df_full = pd.DataFrame(rows)
print(df_full.head())
print("Total samples:", len(df_full))
print(df_full["true_label"].value_counts())


                                          input_text  true_label_id  \
0  Below is an instruction that describes a task,...              1   
1  Below is an instruction that describes a task,...              2   
2  Below is an instruction that describes a task,...             11   
3  Below is an instruction that describes a task,...             12   
4  Below is an instruction that describes a task,...             12   

   true_label  
0    DDoS_UDP  
1   DDoS_ICMP  
2         XSS  
3  Ransomware  
4  Ransomware  
Total samples: 20415
true_label
DDoS_UDP                 1400
DDoS_ICMP                1400
XSS                      1400
Ransomware               1400
SQL_injection            1400
Uploading                1400
Normal                   1400
Vulnerability_scanner    1400
Password                 1400
Backdoor                 1400
DDoS_HTTP                1400
DDoS_TCP                 1400
Port_Scanning            1400
MITM                     1214
Fingerprinting           

In [27]:
from sklearn.model_selection import train_test_split

df_sampled, _ = train_test_split(
    df_full,
    train_size=1000,
    stratify=df_full["true_label"],
    random_state=42
)

print("Sampled shape:", df_sampled.shape)
print(df_sampled["true_label"].value_counts())


Sampled shape: (1000, 3)
true_label
DDoS_HTTP                69
Backdoor                 69
SQL_injection            69
XSS                      69
DDoS_ICMP                69
Vulnerability_scanner    69
Normal                   69
Ransomware               69
Uploading                68
DDoS_TCP                 68
DDoS_UDP                 68
Port_Scanning            68
Password                 68
MITM                     59
Fingerprinting           49
Name: count, dtype: int64


In [28]:
prompts_base = df_sampled["input_text"].tolist()
labels_base_ids = df_sampled["true_label_id"].tolist()
labels_base_str = df_sampled["true_label"].tolist()


In [29]:
from tqdm import tqdm
from sklearn.metrics import f1_score

def run_deepseek_on_prompts(prompts, true_label_ids, max_new_tokens=30):
    preds = []
    for prompt in tqdm(prompts, desc="Evaluating DeepSeek", unit="sample"):
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=max_new_tokens,
                use_cache=True,
            )

        predicted_response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        if "### Response:" in predicted_response:
            response_part = predicted_response.split("### Response:", maxsplit=1)[1].strip()
            pred_id = label_to_multiclass(response_part)
        else:
            pred_id = attack_type_to_id["Normal"]

        preds.append(pred_id)

    f1 = f1_score(true_label_ids, preds, average="macro", zero_division=0)
    return f1, preds


In [ ]:
import re
import numpy as np

rng = np.random.default_rng(42)

# ============================================================
# 1.- Gaussian noise on tcp.dstport
# ============================================================
def drift_noise_tcp_dstport(prompt, sigma_factor=0.1):
    prompt_new = str(prompt)
    pattern = r"(TCP destination port is:\s*)([0-9]+(\.[0-9]+)?)"
    m = re.search(pattern, prompt_new)
    if not m:
        return prompt_new
    val = float(m.group(2))
    sigma = sigma_factor * max(val, 1.0)
    noisy = max(0.0, val + rng.normal(0, sigma))
    replacement = f"{m.group(1)}{noisy:.1f}"
    return re.sub(pattern, replacement, prompt_new, count=1)


# ============================================================
# 2.- Categorical corruption of TCP options
# ============================================================
def drift_corrupt_tcp_options(prompt):
    prompt_new = str(prompt)
    # Match ANY non-space token after the colon
    pattern = r"(TCP options set in the packet are:\s*)(\S+)"
    m = re.search(pattern, prompt_new)
    if not m:
        return prompt_new
    dummy = "DEADBEEFCAFEBABE"
    replacement = f"{m.group(1)}{dummy}"
    return re.sub(pattern, replacement, prompt_new, count=1)


# ============================================================
# 3.- Drop destination port message line
# ============================================================
def drift_drop_dest_port_msg(prompt):
    prompt_new = str(prompt)
    lines = prompt_new.splitlines()
    new_lines = []
    for line in lines:
        if "TCP destination port is:" in line:
            continue   # eliminamos esa línea
        new_lines.append(line)
    return "\n".join(new_lines)


# ============================================================
# 4.- DNS query length +10
# ============================================================
def drift_shift_dns_len(prompt, shift=10.0):
    prompt_new = str(prompt)
    pattern = r"(The length of the DNS query is:\s*)([0-9]+(\.[0-9]+)?)"
    m = re.search(pattern, prompt_new)
    if not m:
        return prompt_new
    val = float(m.group(2))
    new_val = max(0.0, val + shift)
    replacement = f"{m.group(1)}{new_val:.1f}"
    return re.sub(pattern, replacement, prompt_new, count=1)

# ============================================================
# 5.- Drop MQTT msg line
# ============================================================
def drift_drop_mqtt_msg(prompt):
    prompt_new = str(prompt)
    lines = prompt_new.splitlines()
    new_lines = []
    for line in lines:
        if "The MQTT message type is:" in line:
            continue   # eliminamos esa línea
        new_lines.append(line)
    return "\n".join(new_lines)


In [31]:
prompts_base = df_sampled["input_text"].tolist()
labels_base_ids = df_sampled["true_label_id"].tolist()


In [49]:
# Gaussian noise
prompts_noise = [drift_noise_tcp_dstport(p) for p in prompts_base]

# Corruption of tcp.options
prompts_cat = [drift_corrupt_tcp_options(p) for p in prompts_base]

# Drop MQTT message type line
prompts_drop = [drift_drop_dest_port_msg(p) for p in prompts_base]

# DNS query length shift +10
prompts_shift = [drift_shift_dns_len(p, shift=10.0) for p in prompts_base]

# Drop MQTT message type line
prompts_drop_mqtt = [drift_drop_mqtt_msg(p) for p in prompts_base]


In [33]:
f1_base, _ = run_deepseek_on_prompts(prompts_base, labels_base_ids)

Evaluating DeepSeek: 100%|██████████| 1000/1000 [30:43<00:00,  1.84s/sample]


In [50]:
f1_drop_mqtt, _ = run_deepseek_on_prompts(prompts_drop_mqtt, labels_base_ids)

Evaluating DeepSeek: 100%|██████████| 1000/1000 [31:14<00:00,  1.87s/sample]


In [34]:
f1_noise, _ = run_deepseek_on_prompts(prompts_noise, labels_base_ids)
f1_cat,   _ = run_deepseek_on_prompts(prompts_cat, labels_base_ids)
f1_drop,  _ = run_deepseek_on_prompts(prompts_drop, labels_base_ids)
f1_shift, _ = run_deepseek_on_prompts(prompts_shift, labels_base_ids)

Evaluating DeepSeek: 100%|██████████| 1000/1000 [30:11<00:00,  1.81s/sample]


In [53]:
print("\n=== DeepSeek Log Drift Results (N=1000) ===\n")
print(f"Baseline F1:                {f1_base:.4f}")
print(f"Gaussian noise (dstport):   {f1_noise:.4f}      ΔF1 = {f1_noise - f1_base:+.4f}")
print(f"TCP options corruption:     {f1_cat:.4f}        ΔF1 = {f1_cat - f1_base:+.4f}")
print(f"Drop MQTT msg line:         {f1_drop_mqtt:.4f}  ΔF1 = {f1_drop_mqtt - f1_base:+.4f}")
print(f"DNS query len shift (+10):  {f1_shift:.4f}      ΔF1 = {f1_shift - f1_base:+.4f}")
print("\n==========================================\n")



=== DeepSeek Log Drift Results (N=1000) ===

Baseline F1:                0.7121
Gaussian noise (dstport):   0.6612      ΔF1 = -0.0509
TCP options corruption:     0.1210        ΔF1 = -0.5910
Drop MQTT msg line:         0.7023  ΔF1 = -0.0097
DNS query len shift (+10):  0.7043      ΔF1 = -0.0078


